# 03 简化 3DGS（可反传 index_add 版）

In [ ]:
import math, random, shutil
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

assert torch.cuda.is_available()
print('GPU', torch.cuda.get_device_name(0))
DEVICE='cuda'
WORK=Path('/kaggle/working')
DATA=WORK/'gs_data'/'images'; OUT=WORK/'outputs_3dgs'
DATA.mkdir(parents=True, exist_ok=True); OUT.mkdir(exist_ok=True)

srcs=[p for p in Path('/kaggle').rglob('view_*') if p.suffix.lower() in {'.png','.jpg','.jpeg'}]
if len(srcs)<2:
    srcs=[p for p in Path('/kaggle').rglob('*.png') if 'gen_images' in str(p) or 'mv_images' in str(p)][:8]
if len(srcs)<2:
    for i in range(4):
        arr=np.zeros((256,256,3),np.uint8); arr[:,:,i%3]=200; arr[70:190,70:190]=(30,180,90)
        fp=DATA/f'{i:04d}.png'; Image.fromarray(arr).save(fp); srcs.append(fp)
for i,p in enumerate(sorted(set(srcs))[:8]):
    im=Image.open(p).convert('RGB').resize((128,128)); im.save(DATA/f'{i:04d}.png')

imgs=[]
for p in sorted(DATA.glob('*.png')):
    im=Image.open(p).convert('RGB').resize((128,128))
    imgs.append(torch.from_numpy(np.array(im)).float()/255.)
imgs=torch.stack(imgs,0).to(DEVICE)
N,H,W,_=imgs.shape
print('dataset', tuple(imgs.shape))

def look_at(eye):
    eye=np.asarray(eye,float); target=np.zeros(3); up=np.array([0.,1.,0.])
    z=eye-target; z/=np.linalg.norm(z)+1e-8
    x=np.cross(up,z); x/=np.linalg.norm(x)+1e-8
    y=np.cross(z,x); R=np.stack([x,y,z],0); t=-R@eye
    return np.concatenate([R,t[:,None]],1)

cams=[]
for i in range(N):
    az=2*math.pi*i/max(N,1); el=math.radians(15)
    eye=np.array([2.5*math.cos(el)*math.cos(az), 2.5*math.sin(el), 2.5*math.cos(el)*math.sin(az)])
    cams.append(look_at(eye))
fx=fy=0.9*W; cx,cy=W/2,H/2

M=2000
means=torch.nn.Parameter(torch.randn(M,3,device=DEVICE)*0.25)
logit_op=torch.nn.Parameter(torch.zeros(M,device=DEVICE))
colors=torch.nn.Parameter(torch.rand(M,3,device=DEVICE))
opt=torch.optim.Adam([means, logit_op, colors], lr=5e-3)

def render(means, logit_op, colors, Rt):
    R=torch.tensor(Rt[:,:3], device=DEVICE, dtype=means.dtype)
    t=torch.tensor(Rt[:,3], device=DEVICE, dtype=means.dtype)
    pc=means@R.T + t
    z=pc[:,2].clamp(min=1e-3)
    u=fx*(pc[:,0]/z)+cx
    v=fy*(pc[:,1]/z)+cy
    # soft in-bounds mask
    inb=(u>0)&(u<W-1)&(v>0)&(v<H-1)&(z>0.05)
    u=u.clamp(0,W-1); v=v.clamp(0,H-1)
    ui=u.long().clamp(0,W-1); vi=v.long().clamp(0,H-1)
    idx=vi*W+ui
    w=torch.sigmoid(logit_op)*inb.float()
    cols=torch.sigmoid(colors)
    vals=w.unsqueeze(-1)*cols
    flat=torch.zeros(H*W,3, device=DEVICE)
    wflat=torch.zeros(H*W,1, device=DEVICE)
    flat=flat.index_add(0, idx, vals)
    wflat=wflat.index_add(0, idx, w.unsqueeze(-1))
    img=(flat/(wflat+1e-4)).view(H,W,3)
    return img.clamp(0,1)

losses=[]
for step in range(100):
    opt.zero_grad()
    i=random.randrange(N)
    pred=render(means, logit_op, colors, cams[i])
    loss=F.l1_loss(pred, imgs[i]) + 1e-4*(means**2).mean()
    loss.backward(); opt.step()
    losses.append(float(loss.detach()))
    if step%20==0 or step==99:
        print(f'step {step} loss={losses[-1]:.4f}')

plt.figure(figsize=(4,3)); plt.plot(losses); plt.title('L1'); plt.tight_layout(); plt.savefig(OUT/'loss.png'); plt.close()
with torch.no_grad():
    pred0=render(means, logit_op, colors, cams[0])
fig,ax=plt.subplots(1,2,figsize=(6,3))
ax[0].imshow(imgs[0].cpu().numpy()); ax[0].set_title('GT')
ax[1].imshow(pred0.cpu().numpy()); ax[1].set_title('Pred')
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig(OUT/'compare.png'); plt.close()

with torch.no_grad():
    xyz=means.cpu().numpy()
    rgb=(torch.sigmoid(colors).cpu().numpy()*255).astype(np.uint8)
    op=torch.sigmoid(logit_op).cpu().numpy()
keep=op>0.05; xyz,rgb=xyz[keep],rgb[keep]
ply=OUT/'gaussians_as_pointcloud.ply'
with open(ply,'w') as f:
    f.write('ply\nformat ascii 1.0\n')
    f.write(f'element vertex {len(xyz)}\nproperty float x\nproperty float y\nproperty float z\n')
    f.write('property uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n')
    for p,c in zip(xyz,rgb):
        f.write(f'{p[0]} {p[1]} {p[2]} {int(c[0])} {int(c[1])} {int(c[2])}\n')
shutil.make_archive('/kaggle/working/gs_export','zip', OUT)
print('03 DONE', ply, len(xyz))
assert ply.exists() and len(losses)>0